In [ ]:
# Import the basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Load the dataset
from sklearn.datasets import load_breast_cancer

raw = load_breast_cancer()

dataset = pd.DataFrame(data=raw.data, columns=raw.feature_names)
dataset['target'] = raw.target

print("Shape of dataset:", dataset.shape)
dataset.head()

In [ ]:
# Split data into Features (X) and Label (y)
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Split into Training and Testing sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=0
)

print("Training samples:", X_train.shape[0])
print("Testing  samples:", X_test.shape[0])

In [ ]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()

# fit_transform on training data: learns the mean & std, then scales
X_train = sc.fit_transform(X_train)

# transform (NOT fit_transform) on test data: reuse the same mean & std from training
X_test = sc.transform(X_test)

In [ ]:
# Train the SVM Model
# SVC (Support Vector Classifier)
#
# kernel='rbf' -> Radial Basis Function kernel
#   allows SVM to draw curved (non-linear) decision boundaries
#   Works great when data is NOT linearly separable

from sklearn.svm import SVC
classifier = SVC(kernel='rbf', random_state=0)

# The model figures out the best hyperplane to separate classes
classifier.fit(X_train, y_train)

In [ ]:
# Make Predictions on the Test Set
y_pred = classifier.predict(X_test)

print("Predicted: ", y_pred[:10])
print("Actual:    ", y_test[:10])

In [ ]:
# Confusion Matrix
# A confusion matrix tells us:
#   True Positives  (TP): correctly predicted "Benign"
#   True Negatives  (TN): correctly predicted "Malignant"
#   False Positives (FP): predicted Benign, but actually Malignant
#   False Negatives (FN): predicted Malignant, but actually Benign
#
#   Layout:  [ [TN, FP],
#               [FN, TP] ]

from sklearn.metrics import confusion_matrix, accuracy_score

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Accuracy = (TP + TN) / Total predictions
acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {acc * 100:.2f}%")

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred, target_names=['Malignant', 'Benign'])
print("Classification Report:")
print(report)

In [ ]:
# Visualise the Decision Boundary on the TRAINING SET
# Since we have 30 features we can't plot directly in 2D
# PCA reduces 30 dimensions -> 2 dimensions for plotting only
# The actual classifier above still uses all 30 features
from sklearn.decomposition import PCA
from matplotlib.colors import ListedColormap

pca = PCA(n_components=2)
X_train_2d = pca.fit_transform(X_train)
X_test_2d  = pca.transform(X_test)

# Train a second SVM on the 2D data - only used for the decision boundary plot
classifier_2d = SVC(kernel='rbf', random_state=0)
classifier_2d.fit(X_train_2d, y_train)

def plot_decision_boundary(X_set, y_set, title):
    X1, X2 = np.meshgrid(
        np.arange(start=X_set[:, 0].min() - 1, stop=X_set[:, 0].max() + 1, step=0.05),
        np.arange(start=X_set[:, 1].min() - 1, stop=X_set[:, 1].max() + 1, step=0.05)
    )

    # Predict the class for every point on the grid
    # np.c_ stacks the flattened X1 and X2 side-by-side to form (n, 2) input
    Z = classifier_2d.predict(np.c_[X1.ravel(), X2.ravel()])
    Z = Z.reshape(X1.shape)  # reshape back to grid shape for plotting

    plt.figure(figsize=(8, 6))

    # Fill the background with colours based on predictions
    plt.contourf(X1, X2, Z, alpha=0.4,
                 cmap=ListedColormap(['#FF9999', '#99FF99']))

    # Plot the actual data points on top
    for i, j in enumerate(np.unique(y_set)):
        plt.scatter(
            X_set[y_set == j, 0],  # Principal Component 1
            X_set[y_set == j, 1],  # Principal Component 2
            c=['red', 'green'][i],
            label=['Malignant', 'Benign'][i],
            edgecolors='black',
            s=30
        )

    plt.title(title)
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_decision_boundary(X_train_2d, y_train, 'SVM (RBF) - Training Set')

In [ ]:
# Visualise the Decision Boundary on the TEST SET
plot_decision_boundary(X_test_2d, y_test, 'SVM (RBF) - Test Set')

In [ ]:
# Visualise the Test set results
from matplotlib.colors import ListedColormap

X_set, y_set = X_test_2d, y_test

X1, X2 = np.meshgrid(
    np.arange(start = X_set[:, 0].min() - 1,
              stop  = X_set[:, 0].max() + 1,
              step  = 0.05),
    np.arange(start = X_set[:, 1].min() - 1,
              stop  = X_set[:, 1].max() + 1,
              step  = 0.05)
)

plt.contourf(
    X1, X2,
    classifier_2d.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
    alpha = 0.75,
    cmap  = ListedColormap(('red', 'green'))
)

plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())

for i, j in enumerate(np.unique(y_set)):
    plt.scatter(
        X_set[y_set == j, 0],
        X_set[y_set == j, 1],
        c     = ListedColormap(('red', 'green'))(i),
        label = ['Malignant', 'Benign'][i]
    )

plt.title('SVM (Test set)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.show()